# Malaria Diagnosis - Improved CNN · Owner: **Apoh Prince Eldrige**

I'm Apoh Prince Eldrige, and my role was to take Dan's baseline and genuinely push it. My starting assumption was that the baseline's real weakness would be overfitting, a plain CNN tends to memorise the training cells rather than learn the disease, so I built the improved model squarely around that problem: a third convolution block for more representational capacity, batch normalisation for stable training, dropout, and data augmentation to force the network to generalise instead of memorise.

I treated my seven experiments as a ladder rather than a single jump. I add depth first, then batch-norm, then augmentation, and only then do I tune the regularisation strength. Each rung isolates one ingredient, so I can point to precisely what closed the train-validation gap instead of waving at a pile of changes. My working hypothesis is that augmentation will do most of the heavy lifting, and the overfit_gap column is where I intend to confirm or kill that idea.

## ⚡ Notes to ourselves before running this

We learned the hard way that the free Colab GPU runs out before you can train five models seven times each, so we redesigned the notebook around that problem. A few decisions worth explaining:

- **We use all 27,558 images** (`SAMPLE_FRACTION = 1.0`). At first we were tempted to train on a smaller slice to save time, but we realised that using less data is exactly what makes a model overfit, which is the opposite of what we want. More data is the cheapest way to generalise better, so we kept all of it.
- **We switched to a `tf.data` pipeline** (`image_dataset_from_directory` with cache + prefetch). Our first version used the older `ImageDataGenerator` and it was painfully slow, the GPU was sitting idle waiting for images. Caching and prefetching keep the GPU busy, which is what made full-data training actually affordable.
- **We added EarlyStopping and ReduceLROnPlateau.** We put these in specifically to control fitting: EarlyStopping (with best-weight restore) stops training the moment the validation loss stops improving so the model can't keep memorising the training set, and ReduceLROnPlateau drops the learning rate when progress stalls so the model doesn't get stuck underfitting.
- **We made it resumable.** Each experiment's results are written to `all_experiment_results.csv` the instant it finishes, and the notebook skips anything already in that file. This way, when Colab disconnects, we just re-run everything and it carries on from where it stopped instead of throwing away hours of work. Set `USE_DRIVE_CKPT = True` to keep that file on Drive so it even survives a full runtime reset.

## 1. Why this problem matters to us

Malaria is caused by *Plasmodium* parasites spread by *Anopheles* mosquitoes, and it still kills hundreds of thousands of people a year, mostly young children. The way it is normally diagnosed is a technician looking at a stained blood smear under a microscope and counting infected cells by hand. We chose to work on this because that manual process is slow, needs a trained expert, and isn't available everywhere it's needed, which is exactly the kind of bottleneck a trained model could help with. Our objective in this notebook is to automate the "parasitized vs uninfected" decision on single-cell images.

## 2. Configuration & environment

We keep all the settings we tune in one place so we don't have magic numbers scattered through the notebook. Make sure the runtime is set to **GPU** (*Runtime → Change runtime type → GPU*), we check for it below because without a GPU the training times are not realistic.

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus if gpus else "NONE - enable a GPU runtime!")

In [ ]:
# ---------------- RUN SETTINGS ----------------
SAMPLE_FRACTION = 1.0      # use ALL images (full accuracy). Lower only for a quick smoke test.
MAX_EPOCHS      = 25       # upper bound; EarlyStopping usually stops earlier
PATIENCE        = 5        # EarlyStopping patience on val_loss
MIXED_PRECISION = True     # faster + less GPU memory
RESET_RESULTS   = False    # True once to wipe saved results and start over
USE_DRIVE_CKPT  = True     # ON: results CSV is saved to Google Drive so progress survives a runtime disconnect

IMG_SIZE   = (128, 128)
BATCH_SIZE = 32
MODEL_TAG   = "ImprovedCNN"          # this notebook's model - keep UNIQUE per member
RESULTS_CSV = f"results_{MODEL_TAG}.csv"
print(f"data={int(SAMPLE_FRACTION*100)}% | max_epochs={MAX_EPOCHS} | patience={PATIENCE} | batch={BATCH_SIZE}")

In [ ]:
import os, random, shutil, zipfile, urllib.request, time, warnings, gc
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc)
from tensorflow.keras import layers, Model, Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     BatchNormalization, GlobalAveragePooling2D, Input, Rescaling,
                                     RandomFlip, RandomRotation, RandomZoom, RandomTranslation)
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K, mixed_precision
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.utils import image_dataset_from_directory

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
if MIXED_PRECISION and gpus:
    mixed_precision.set_global_policy('mixed_float16'); print("Mixed precision: ON")
else:
    print("Mixed precision: OFF")

In [ ]:
# ==================== CHECKPOINT / RESUME (survives runtime disconnects) ====================
# WHY: Colab wipes the local VM when the runtime disconnects. Saving the results CSV to Google
# Drive means every FINISHED experiment is kept. On reconnect, just Run-all again: run_experiment()
# reads this CSV and SKIPS experiments already done (see _done/_save below), so you resume instead
# of starting over. Set USE_DRIVE_CKPT=False in RUN SETTINGS only if you are NOT on Colab.
if USE_DRIVE_CKPT:
    try:
        from google.colab import drive
        drive.mount('/content/drive')                       # one-time auth per session
        CKPT_DIR = '/content/drive/MyDrive/malaria_ckpt'
        os.makedirs(CKPT_DIR, exist_ok=True)
        RESULTS_CSV = os.path.join(CKPT_DIR, f'results_{MODEL_TAG}.csv')
        print('Checkpoint ON -> progress saved to:', RESULTS_CSV)
        # TIP: point CKPT_DIR at a SHARED Drive folder so the compiler can read everyone's results_*.csv
        if os.path.exists(RESULTS_CSV):
            print('   Found existing checkpoint - finished experiments will be skipped on resume.')
        else:
            print('   No checkpoint yet - a fresh one will be created as experiments finish.')
    except Exception as e:
        print('Drive mount failed (not on Colab?) - falling back to local CSV:', e)
        print('   WARNING: local CSV is lost if the runtime disconnects.')
else:
    print('Checkpoint OFF - results CSV stays on the local VM and is lost on disconnect.')


## 3. Dataset - downloaded straight from the NIH link

We deliberately download the dataset from the official NIH URL instead of mounting it from someone's Google Drive. We did this so that anyone in the group (or the facilitator) can run the notebook from scratch without needing access to our personal drives, it makes the work reproducible. The dataset has 27,558 single-cell images, evenly split between `Parasitized` and `Uninfected`, which is convenient because a balanced dataset means accuracy is actually a meaningful metric for us.

In [ ]:
DATA_URL = "https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip"
ZIP_PATH, RAW_DIR = "cell_images.zip", "cell_images"
if not os.path.exists(RAW_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Downloading from NIH (~337 MB) ..."); urllib.request.urlretrieve(DATA_URL, ZIP_PATH); print("done.")
    print("Extracting ...");
    with zipfile.ZipFile(ZIP_PATH,"r") as z: z.extractall(".")
    print("done.")
else:
    print("Dataset already present.")
if os.path.isdir(os.path.join(RAW_DIR,"cell_images")): RAW_DIR = os.path.join(RAW_DIR,"cell_images")
for c in ["Parasitized","Uninfected"]:
    print(f"  {c}: {len(os.listdir(os.path.join(RAW_DIR,c)))} images")

### 3.1 Splitting the data 70/15/15

We split the images once into training, validation and test sets and then reuse that exact same split for every model. We thought about letting each model make its own split, but then any difference in results could just be down to luck in the split rather than the model itself. Fixing the split (and the random seed) means that when VGG16 beats the baseline, we can actually attribute that to the model. We use 70% for training, 15% to tune and watch for overfitting, and a final 15% that we never touch until the end.

In [ ]:
BASE = "dataset"
if not os.path.exists(BASE):
    for split in ["train","val","test"]:
        for c in ["Parasitized","Uninfected"]:
            os.makedirs(os.path.join(BASE,split,c), exist_ok=True)
    for c in ["Parasitized","Uninfected"]:
        files = [f for f in os.listdir(os.path.join(RAW_DIR,c)) if f.lower().endswith(".png")]
        random.shuffle(files)
        if SAMPLE_FRACTION < 1.0: files = files[:int(len(files)*SAMPLE_FRACTION)]
        n=len(files); ntr=int(n*0.70); nva=int(n*0.15)
        for split,fl in {"train":files[:ntr],"val":files[ntr:ntr+nva],"test":files[ntr+nva:]}.items():
            for f in fl: shutil.copy(os.path.join(RAW_DIR,c,f), os.path.join(BASE,split,c,f))
    print("Split created.")
else:
    print("Split exists (delete the 'dataset' folder to rebuild).")
for split in ["train","val","test"]:
    print(split, {c: len(os.listdir(os.path.join(BASE,split,c))) for c in ["Parasitized","Uninfected"]})

### 3.2 The input pipeline (built once and shared)

This was our biggest performance fix. We build the train/validation/test datasets a single time and reuse them across all 35 experiments, instead of rebuilding a slow generator every run. We `cache()` so the images only get read and decoded from disk once, and `prefetch()` so the next batch is being prepared on the CPU while the GPU trains on the current one. We also keep the class order fixed and leave the test set unshuffled, otherwise our predictions wouldn't line up with the true labels when we score them.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = ["Uninfected","Parasitized"]
def _ds(split, shuffle):
    return image_dataset_from_directory(os.path.join(BASE,split), labels="inferred", label_mode="binary",
        class_names=["Uninfected","Parasitized"], image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED)
train_ds_raw = _ds("train", True)
val_ds_raw   = _ds("val",   False)
test_ds_raw  = _ds("test",  False)

# y_true for the test set (order preserved because shuffle=False)
y_test = np.concatenate([y.numpy() for _,y in test_ds_raw]).ravel().astype(int)

train_ds = train_ds_raw.cache().prefetch(AUTOTUNE)
val_ds   = val_ds_raw.cache().prefetch(AUTOTUNE)
test_ds  = test_ds_raw.cache().prefetch(AUTOTUNE)
print("Pipeline ready. Test labels:", np.bincount(y_test))

## 4. The training engine (shared so every model is judged the same way)

We wrote one training/evaluation function that every member's model goes through, because if we each scored our own model our own way the comparison would be meaningless. A couple of choices we made here on purpose:

- **Augmentation and preprocessing are baked into the model as layers.** We flip, rotate, zoom and shift the training images so the model sees more variety and is less likely to memorise specific cells. We put the backbone-specific preprocessing in as a layer too, so each pretrained model gets the exact input format it was originally trained on.
- **EarlyStopping + ReduceLROnPlateau on every run.** These are our main tools against over- and underfitting, so we apply them everywhere rather than per-model.
- **We log a `train_acc`, `val_acc` and `overfit_gap`** for every experiment. We added the gap column specifically so we have hard evidence for the overfitting discussion in the report instead of just eyeballing the curves.
- **`plot_best` retrains only the best configuration** of a model to draw its plots. We did it this way so the diagnostic plots still appear even after a disconnect, without paying to retrain all seven runs.

In [ ]:
# Augmentation pipeline (active only during training) and per-backbone preprocessing
data_augment = Sequential([
    RandomFlip("horizontal_and_vertical"),
    RandomRotation(0.10),
    RandomZoom(0.10),
    RandomTranslation(0.10, 0.10),
], name="augment")

from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_pre
BACKBONES = {"VGG16":(VGG16,vgg_pre), "ResNet50":(ResNet50,resnet_pre), "MobileNetV2":(MobileNetV2,mobilenet_pre)}

def _stem(cfg, inp):
    """Augment (if requested) then apply the right preprocessing as layers."""
    x = data_augment(inp) if cfg.get("augment") else inp
    if cfg.get("backbone"):
        x = layers.Lambda(BACKBONES[cfg["backbone"]][1])(x)
    else:
        x = Rescaling(1./255)(x)
    return x

def out_dense(x):
    return Dense(1, activation="sigmoid", dtype="float32")(x)

In [ ]:
def build_improved(cfg, input_shape):
    inp = Input(input_shape); x = _stem(cfg, inp)
    x = Conv2D(32,(3,3),activation="relu",padding="same")(x)
    if cfg["bn"]: x = BatchNormalization()(x)
    x = MaxPooling2D(2,2)(x)
    x = Conv2D(64,(3,3),activation="relu",padding="same")(x)
    if cfg["bn"]: x = BatchNormalization()(x)
    x = MaxPooling2D(2,2)(x)
    if cfg["blocks"]>=3:
        x = Conv2D(128,(3,3),activation="relu",padding="same")(x)
        if cfg["bn"]: x = BatchNormalization()(x)
        x = MaxPooling2D(2,2)(x)
    x = Flatten()(x); x = Dense(cfg["dense"],activation="relu")(x); x = Dropout(cfg["dropout"])(x)
    return Model(inp, out_dense(x))

In [ ]:
RESULTS = []
if RESET_RESULTS and os.path.exists(RESULTS_CSV): os.remove(RESULTS_CSV); print("Results reset.")
if os.path.exists(RESULTS_CSV):
    RESULTS = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"Resumed {len(RESULTS)} finished experiment(s).")
else:
    print("No previous results - starting fresh.")

def _done(model_name, exp_id):
    return any(r["model"]==model_name and r["experiment"]==exp_id for r in RESULTS)
def _save(): pd.DataFrame(RESULTS).to_csv(RESULTS_CSV, index=False)

def evaluate(model):
    y_prob = model.predict(test_ds, verbose=0).ravel().astype("float32")
    y_pred = (y_prob>=0.5).astype(int)
    return {"accuracy":accuracy_score(y_test,y_pred), "precision":precision_score(y_test,y_pred,zero_division=0),
            "recall":recall_score(y_test,y_pred,zero_division=0), "f1":f1_score(y_test,y_pred,zero_division=0),
            "y_prob":y_prob, "y_pred":y_pred}

def _train(cfg):
    K.clear_session(); gc.collect()
    model = cfg["build_fn"](cfg, IMG_SIZE+(3,))
    model.compile(optimizer=cfg["opt"](), loss="binary_crossentropy", metrics=["accuracy"])
    cbs = [EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
           ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)]
    t0=time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=min(cfg["epochs"],MAX_EPOCHS), callbacks=cbs, verbose=0)
    return model, hist, evaluate(model), round(time.time()-t0,1)

def run_experiment(owner, model_name, cfg):
    if _done(model_name, cfg["name"]):
        print(f"[{model_name}] {cfg['name']}: already done - skipped"); return
    model, hist, m, secs = _train(cfg)
    tr_acc = float(hist.history["accuracy"][-1]); va_acc = float(hist.history["val_accuracy"][-1])
    RESULTS.append({"owner":owner,"model":model_name,"experiment":cfg["name"],"config":cfg["desc"],
        "accuracy":round(m["accuracy"],4),"precision":round(m["precision"],4),"recall":round(m["recall"],4),
        "f1":round(m["f1"],4),"train_acc":round(tr_acc,4),"val_acc":round(va_acc,4),
        "overfit_gap":round(tr_acc-va_acc,4),"epochs_run":len(hist.history["loss"]),"train_sec":secs})
    _save()
    print(f"[{model_name}] {cfg['name']}: acc={m['accuracy']:.4f} f1={m['f1']:.4f} gap={tr_acc-va_acc:+.3f} (saved)")
    del model; K.clear_session(); gc.collect()

def results_table(model_name):
    df = pd.DataFrame([r for r in RESULTS if r["model"]==model_name])
    cols=["experiment","config","accuracy","precision","recall","f1","train_acc","val_acc","overfit_gap","epochs_run","train_sec"]
    return df[[c for c in cols if c in df.columns]]

def plot_best(model_name, exps):
    rows=[r for r in RESULTS if r["model"]==model_name]
    if not rows: print("No results yet for", model_name); return
    best=max(rows, key=lambda r:r["f1"])
    cfg=next(e for e in exps if e["name"]==best["experiment"])
    print(f"Retraining best config of {model_name} for plots: {cfg['name']}")
    model,hist,m,_=_train(cfg)
    fig,ax=plt.subplots(1,4,figsize=(22,4.5))
    ax[0].plot(hist.history["loss"],label="train"); ax[0].plot(hist.history["val_loss"],label="val")
    ax[0].set_title(f"{model_name} - Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist.history["accuracy"],label="train"); ax[1].plot(hist.history["val_accuracy"],label="val")
    ax[1].set_title(f"{model_name} - Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    cm=confusion_matrix(y_test, m["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Uninfected","Parasitized"]).plot(ax=ax[2], colorbar=False)
    ax[2].set_title(f"{model_name} - Confusion matrix")
    fpr,tpr,_=roc_curve(y_test, m["y_prob"]); roc_auc=auc(fpr,tpr)
    ax[3].plot(fpr,tpr,label=f"AUC = {roc_auc:.3f}"); ax[3].plot([0,1],[0,1],"k--")
    ax[3].set_title(f"{model_name} - ROC"); ax[3].set_xlabel("FPR"); ax[3].set_ylabel("TPR"); ax[3].legend()
    plt.tight_layout(); plt.show()
    print(f"Best F1={best['f1']:.4f} | AUC={roc_auc:.4f} | train-val gap={best.get('overfit_gap')}")
    del model; K.clear_session(); gc.collect(); return roc_auc

## 6. Model 2 - Improved CNN · Owner: **Apoh Prince Eldrige**

I, Apoh Prince Eldrige, built this improved CNN to beat the baseline with a deeper, better-regularised network. I added a third convolution block (more capacity to learn shapes), batch normalisation (to make training more stable and faster), dropout (to fight overfitting) and data augmentation. I designed my seven experiments as a ladder, first just adding depth, then batch-norm, then augmentation, then tuning the regularisation, so that I can point to exactly which ingredient gave the improvement and by how much. I expected augmentation in particular to close the train-validation gap I saw in the baseline.

In [ ]:
improved_exps=[
 {"name":"E1 2blk noBN noAug","desc":"2 blocks, no BN, no aug (control)","blocks":2,"bn":False,"dense":128,"dropout":0.3,"epochs":18,"augment":False,"opt":lambda:Adam(1e-3),"build_fn":build_improved},
 {"name":"E2 3blk noBN","desc":"3 blocks, no BN","blocks":3,"bn":False,"dense":128,"dropout":0.3,"epochs":18,"augment":False,"opt":lambda:Adam(1e-3),"build_fn":build_improved},
 {"name":"E3 3blk +BN","desc":"3 blocks + BatchNorm","blocks":3,"bn":True,"dense":128,"dropout":0.3,"epochs":18,"augment":False,"opt":lambda:Adam(1e-3),"build_fn":build_improved},
 {"name":"E4 +BN +Aug","desc":"3 blocks + BN + augmentation","blocks":3,"bn":True,"dense":128,"dropout":0.3,"epochs":22,"augment":True,"opt":lambda:Adam(1e-3),"build_fn":build_improved},
 {"name":"E5 +Aug drop0.5","desc":"+ aug, dropout 0.5","blocks":3,"bn":True,"dense":128,"dropout":0.5,"epochs":22,"augment":True,"opt":lambda:Adam(1e-3),"build_fn":build_improved},
 {"name":"E6 +Aug lr1e-4","desc":"+ aug, LR 1e-4, dense 256","blocks":3,"bn":True,"dense":256,"dropout":0.4,"epochs":25,"augment":True,"opt":lambda:Adam(1e-4),"build_fn":build_improved},
 {"name":"E7 wider256 +Aug","desc":"Wider 256 + aug, dropout 0.5","blocks":3,"bn":True,"dense":256,"dropout":0.5,"epochs":25,"augment":True,"opt":lambda:Adam(1e-3),"build_fn":build_improved},
]
for e in improved_exps: run_experiment("Apoh Prince Eldrige","Improved CNN", e)

In [ ]:
results_table("Improved CNN")

In [ ]:
plot_best("Improved CNN", improved_exps)